<a href="https://colab.research.google.com/github/anseohyeonn/bigdata/blob/main/week05_5874787_%EC%95%88%EC%84%9C%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름: 안서현**
**학번: 5874787**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [ ]:
# 답안
# 사용자 정의 Dataset 클래스
class MyDataset(Dataset):
    def __init__(self, X, y):
        # 입력 데이터 X와 정답 데이터 y를 저장한다.
        self.X = X
        self.y = y

    def __len__(self):
        # 전체 데이터의 개수를 반환한다.
        return len(self.X)

    def __getitem__(self, idx):
        # idx번째 입력 데이터와 정답 데이터를 반환한다.
        return self.X[idx], self.y[idx]


# 데이터 생성: 데이터 30개, 특성 4개
X = torch.randn(30, 4)
y = torch.randn(30, 1)

# Dataset 객체 생성
dataset = MyDataset(X, y)

print("전체 데이터 개수:", len(dataset))
print("입력 데이터 shape:", X.shape)
print("정답 데이터 shape:", y.shape)
print("첫 번째 입력:", dataset[0][0])
print("첫 번째 정답:", dataset[0][1])

전체 데이터 개수: 30
입력 데이터 shape: torch.Size([30, 4])
정답 데이터 shape: torch.Size([30, 1])
첫 번째 입력: tensor([-1.1258, -1.1524, -0.2506, -0.4339])
첫 번째 정답: tensor([1.1404])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [ ]:
# 답안
# Dataset을 DataLoader에 넣어 7개씩 묶는다.
loader = DataLoader(dataset, batch_size=7, shuffle=False)

# 전체 batch 개수 출력
print("batch 개수:", len(loader))

# 각 batch의 입력과 정답 shape 출력
for i, (bx, by) in enumerate(loader):
    print(f"batch {i + 1}: X shape = {tuple(bx.shape)}, "
          f"y shape = {tuple(by.shape)}")

batch 개수: 5
batch 1: X shape = (7, 4), y shape = (7, 1)
batch 2: X shape = (7, 4), y shape = (7, 1)
batch 3: X shape = (7, 4), y shape = (7, 1)
batch 4: X shape = (7, 4), y shape = (7, 1)
batch 5: X shape = (2, 4), y shape = (2, 1)


마지막 batch의 크기가 다른 이유
전체 데이터는 30개이고, batch size는 7이다.

30 ÷ 7 = 4 나머지 2

앞의 4개 batch에는 각각 7개씩 들어간다.

마지막에는 남은 데이터 2개가 들어간다.

따라서 총 batch 개수는 5개이고, 마지막 batch의 크기는 2개다. 기본 설정인 drop_last=False에서는 남은 데이터도 버리지 않고 사용한다.

## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


**답:**

(7, 4)

입력 데이터 7개, 각 데이터의 특성 4개

(7, 1)

정답 데이터 7개, 각 데이터의 정답 1개

(2, 4)

마지막 batch의 입력 데이터 2개, 각 데이터의 특성 4개

(2, 1)

마지막 batch의 정답 데이터 2개, 각 데이터의 정답 1개

정리하면, 첫 번째 숫자는 한 번에 처리하는 데이터 개수이고, 두 번째 숫자는 데이터 하나에 들어 있는 특성 수 또는 정답 수다. 마지막 batch는 남은 데이터가 2개라서 첫 번째 숫자만 2로 바뀐다.

## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [ ]:
# 답안
# DataLoader 생성
loader = DataLoader(dataset, batch_size=7, shuffle=True)

# 1. 선형 모델 생성: 입력 특성 4개 -> 출력 1개
model = nn.Linear(4, 1)

# 손실 함수와 옵티마이저 설정
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 학습 횟수 설정
epochs = 50

# 학습 루프
for epoch in range(epochs):
    total_loss = 0.0

    for bx, by in loader:
        # 1단계: 예측
        pred = model(bx)

        # 2단계: 손실 계산
        loss = loss_fn(pred, by)

        # 3단계: 기울기 초기화
        optimizer.zero_grad()

        # 4단계: 역전파
        loss.backward()

        # 5단계: 파라미터 수정
        optimizer.step()

        # batch별 손실을 데이터 개수만큼 누적
        total_loss += loss.item() * bx.shape[0]

    # epoch의 평균 손실 계산
    avg_loss = total_loss / len(dataset)

    # epoch마다 평균 손실 출력
    print(f"epoch {epoch + 1:2d}, 평균 손실: {avg_loss:.4f}")

epoch  1, 평균 손실: 2.3313
epoch  2, 평균 손실: 2.0603
epoch  3, 평균 손실: 1.8930
epoch  4, 평균 손실: 1.7550
epoch  5, 평균 손실: 1.6639
epoch  6, 평균 손실: 1.5839
epoch  7, 평균 손실: 1.5143
epoch  8, 평균 손실: 1.4892
epoch  9, 평균 손실: 1.4530
epoch 10, 평균 손실: 1.4368
epoch 11, 평균 손실: 1.4272
epoch 12, 평균 손실: 1.4077
epoch 13, 평균 손실: 1.3885
epoch 14, 평균 손실: 1.3824
epoch 15, 평균 손실: 1.3725
epoch 16, 평균 손실: 1.3751
epoch 17, 평균 손실: 1.3599
epoch 18, 평균 손실: 1.3567
epoch 19, 평균 손실: 1.3585
epoch 20, 평균 손실: 1.3658
epoch 21, 평균 손실: 1.3537
epoch 22, 평균 손실: 1.3494
epoch 23, 평균 손실: 1.3448
epoch 24, 평균 손실: 1.3463
epoch 25, 평균 손실: 1.3443
epoch 26, 평균 손실: 1.3439
epoch 27, 평균 손실: 1.3475
epoch 28, 평균 손실: 1.3399
epoch 29, 평균 손실: 1.3404
epoch 30, 평균 손실: 1.3374
epoch 31, 평균 손실: 1.3417
epoch 32, 평균 손실: 1.3451
epoch 33, 평균 손실: 1.3455
epoch 34, 평균 손실: 1.3422
epoch 35, 평균 손실: 1.3411
epoch 36, 평균 손실: 1.3502
epoch 37, 평균 손실: 1.3423
epoch 38, 평균 손실: 1.3469
epoch 39, 평균 손실: 1.3445
epoch 40, 평균 손실: 1.3413
epoch 41, 평균 손실: 1.3365
epoch 42, 평균 손실: